In [ ]:
!git clone https://github.com/evagogua/denoiseML

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
import pandas as pd
from datasets import Dataset
from datasets import load_dataset
import random
import numpy as np
from tqdm.auto import tqdm
import re

# Перевод датасета на русский язык

In [ ]:
!wget https://raw.githubusercontent.com/clinc/oos-eval/refs/heads/master/data/data_full.json

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-ru")

In [ ]:
with open('data_full.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def translate(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    translated = model.generate(
        **inputs,
        max_length=512,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(translated[0], skip_special_tokens=True)

hf_dataset = Dataset.from_list([
    {"text": i[0], "label": i[1]} for item in data.values() for i in item
])

ru_data = Dataset.from_list([
    {
        "text": re.sub(r'[^\w\s]', '', translate(item["text"])).lower(),
        "classification_labels": item["label"]
    }
    for item in tqdm(hf_dataset, total=len(hf_dataset), desc="Перевод")
])

In [ ]:
with open('clean_data.json', 'w', encoding='utf-8') as f:
    json.dump(ru_data.to_list(), f, ensure_ascii=False, indent=2)

# Зашумление датасета


In [ ]:
with open('/content/denoiseML/data/clean_data.json', 'r', encoding='utf-8') as f:
    clean_data = json.load(f)

print(f"Загружено {len(clean_data)} чистых примеров")
print(f"Пример: {clean_data[0]}")

In [ ]:
from datasets import load_dataset

print("\nЗагрузка шумовых фраз из датасета диалогов...")
noise = load_dataset("Den4ikAI/russian_dialogues")

In [ ]:
noise_frases = []
counter = 0
for item in noise['train']:
  # Добавляем вопрос, если он есть и достаточно короткий
  if item['question'] is not None:
    if len(item['question'].split()) < 9:
      noise_frases.append(item['question'])
      counter+=1
  # Добавляем ответ, если он есть и достаточно короткий
    if item['answer'] is not None:
      if len(item['answer'].split()) < 9:
        noise_frases.append(item['answer'])
  # Ограничиваем количество фраз
  if counter == 300000:
    break
random.shuffle(noise_frases)
print(f"Загружено {len(noise_frases)} шумовых фраз")

In [ ]:
from collections import defaultdict

# Создаём словарь, где ключ - длина фразы в словах, значение - список фраз
lengths = defaultdict(list)
for item in noise_frases:
  lengths[len(item.split())].append(item)

In [ ]:
from typing import Sequence, TypeVar, List

T = TypeVar("T")

def insert_many_random(lst: Sequence[T], inserts: Sequence[T], seed: int | None = None) -> List[T]:
    rnd = random.Random(seed)
    n = len(lst)
    k = len(inserts)

    positions = sorted(rnd.sample(range(n + 1), k))

    out: List[T] = []
    last = 0
    newlabel = []
    for pos, ins in zip(positions, inserts):
        new_ins = ins.split()
        out.extend(lst[last:pos])
        newlabel.extend(["0"]*(pos-last)+["N"]*len(new_ins))
        out.extend(new_ins)
        last = pos
    newlabel += ["0"]*(n-last)
    out.extend(lst[last:])
    return " ".join(out), newlabel

print(insert_many_random(["1", "2", "3", "4", "5"], ["A", "B", "C"], seed=42))

In [ ]:
import re

# Создание шумного датасета
seed = 1337
noise_data = []
for item in clean_data:
  oldword = item["text"].split()
  needed = []
  numbers = np.array([0, 1, 2, 3, 4])
  p = np.array([0.2, 0.25, 0.25, 0.15, 0.15])
  number = 100
  while number > len(oldword):
    number = np.random.choice(numbers, size=1, p=p)[0]
  choices = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
  p1 = np.array([0.2, 0.1, 0.1, 0.15, 0.15, 0.15, 0.05, 0.05, 0.05])
  length = 10
  while not lengths[length]:
    length = np.random.choice(choices, size=1, p=p1)[0]
  if number != 0:
    for i in range(number):
      newsmth = None
      while newsmth is None:
        newsmth = re.sub(r'[^\w\s]', '', lengths[length].pop()).lower()
      needed.append(newsmth)
    words, labels = insert_many_random(oldword, needed, seed=seed)
    noise_data.append({'text': words.split(), 'denoise_labels': labels, 'classification_labels': item["classification_labels"]})
  else:
    noise_data.append({'text': oldword, 'denoise_labels': ["0"]*len(oldword), 'classification_labels': item["classification_labels"]})

In [ ]:
noise_ds = Dataset.from_list(noise_data)
noise_ds[0]

In [ ]:
with open('noise_data_all.json', 'w', encoding='utf-8') as f:
    json.dump(noise_ds.to_list(), f, ensure_ascii=False, indent=2)

## Статистика

In [ ]:
with open('/content/denoiseML/data/noise_data_all.json', 'r', encoding='utf-8') as f:
    noise_data = json.load(f)

In [ ]:
amounts = []
for item in noise_data:
  counter = 0
  for ch in item['denoise_labels']:
    if ch == "N":
      counter+=1
  amounts.append(counter)

In [ ]:
statistics = defaultdict(int)
for item in amounts:
  if item == 0:
    statistics["0"]+=1
  elif item <= 4:
    statistics["1-4"]+=1
  elif item <= 8:
    statistics["5-8"]+=1
  elif item <= 12:
    statistics["9-12"]+=1
  elif item <= 16:
    statistics["13-16"]+=1
  else:
    statistics["16+"]+=1
# print("\n".join([f"{k}: {v}" for k, v in statistics.items()]))
[print(f"{key}: {statistics[key]}") for key in sorted(list(statistics.keys()))]
print(sum(amounts)/len(amounts))

In [ ]:
import matplotlib.pyplot as plt

group_order = ["0", "1-4", "5-8", "9-12", "13-16", "16+"]

values = [statistics[key] for key in group_order]

plt.figure(figsize=(10, 5))
plt.bar(group_order, values)
plt.xlabel("Количество шума")
plt.ylabel("Количество предложений")
plt.title("Количество шума на предложение")
plt.show()

In [ ]:
posstatistics = defaultdict(int)
for item in noise_data:
  for i in range(len(item['denoise_labels'])):
    if item['denoise_labels'][i] == "N":
      if i==len(item['denoise_labels']):
        posstatistics[-1]+=1
      else:
        posstatistics[i]+=1
print("\n".join([f"{k}: {v}" for k, v in posstatistics.items()]))

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(sorted(list(posstatistics.keys())), [posstatistics[key] for key in sorted(list(posstatistics.keys()))])
plt.xlabel("Позиция")
plt.ylabel("Количество предложений")
plt.title("Позиция шума в предложении")
plt.show()

In [ ]:
with open('noise_data_all.json', 'w', encoding='utf-8') as f:
    json.dump(noise_ds.to_list(), f, ensure_ascii=False, indent=2)

# Выделение поддатасетов

In [ ]:
banking_labels = ['transfer', 'transactions', 'balance', 'freeze_account', 'pay_bill', 'bill_balance', 'bill_due', 'interest_rate', 'routing_number', 'minimum_payment', 'order_checks', 'pin_change', 'report_fraud', 'account_blocked', 'spending_history']
credit_cards_labels = ['credit_score', 'report_lost_card', 'credit_limit', 'rewards_balance', 'new_card', 'application_status', 'card_declined', 'international_fees', 'apr', 'redeem_rewards', 'credit_limit_change', 'damaged_card', 'replacement_card_duration', 'improve_credit_score', 'expiration_date']
travel_labels = ['book_flight', 'book_hotel', 'car_rental', 'travel_suggestion', 'travel_alert', 'travel_notification', 'carry_on', 'timezone', 'vaccines', 'flight_status', 'international_visa', 'lost_luggage', 'plug_type', 'exchange_rate']
work_labels = ['direct_deposit', 'pto_request', 'taxes', 'payday', 'pto_balance', 'pto_request_status', 'next_holiday', 'insurance', 'insurance_change', 'schedule_meeting', 'pto_used', 'meeting_schedule', 'income']

In [ ]:
banking_noise_dataset = [item for item in noise_ds if item.get('classification_labels') in banking_labels]
credit_cards_noise_dataset = [item for item in noise_ds if item.get('classification_labels') in credit_cards_labels]
travel_noise_dataset = [item for item in noise_ds if item.get('classification_labels') in travel_labels]
work_noise_dataset = [item for item in noise_ds if item.get('classification_labels') in work_labels]

In [ ]:
import random

banking_credit_noise_dataset = banking_noise_dataset + credit_cards_noise_dataset
random.shuffle(banking_credit_noise_dataset)
travel_work_noise_dataset = travel_noise_dataset + work_noise_dataset
random.shuffle(travel_work_noise_dataset)
bctw_noise_dataset = banking_noise_dataset + credit_cards_noise_dataset + travel_noise_dataset + work_noise_dataset
random.shuffle(bctw_noise_dataset)

In [ ]:
with open('noise_data_banking_credit.json', 'w', encoding='utf-8') as f:
    json.dump(banking_credit_noise_dataset, f, ensure_ascii=False, indent=2)
with open('noise_data_travel_work.json', 'w', encoding='utf-8') as f:
    json.dump(banking_credit_noise_dataset, f, ensure_ascii=False, indent=2)
with open('noise_data_bctw.json', 'w', encoding='utf-8') as f:
    json.dump(bctw_noise_dataset, f, ensure_ascii=False, indent=2)